# Laboratorio visual del ETL — Business Intelligence & Analytics

**Objetivo:** entender, paso a paso, qué hace el módulo `src/etl/procesar_datos.py` del TP: fuentes Excel → limpieza → unión → notas → alertas → tablas analíticas.

**No copiamos ni reescribimos la lógica:** importamos las funciones reales del proyecto y mostramos resultados intermedios. Si cambia el script, este laboratorio utiliza su implementación actual al reiniciar el entorno y volver a clonar.

**Orden:** ejecutar las celdas de arriba hacia abajo. La rama elegida es `matias`; no modifica `main`. Este notebook lee los Excel originales, los cuales **no están guardados en GitHub**. Las transformaciones intermedias se mantienen en memoria y al final se puede ejecutar el ETL completo.

> Las cifras y gráficas dependen de los archivos que cargues. Los ejemplos de vista previa muestran un máximo de algunas filas para no saturar Colab.

## 0. Preparar Colab y clonar el repositorio privado

**Antes de ejecutar:** en Colab abrí el panel de **Secrets** (ícono de llave), creá `GITHUB_TOKEN` con un token personal de GitHub de acceso **solo lectura** a `bi-analytics-tp`, y permití el acceso del notebook al secret. No escribas ni publiques el token en una celda.

El código lo pasa como autenticación temporal a Git. No queda escrito en la URL remota, en el notebook ni en el resultado de las celdas. Si ya tenés el repositorio clonado en `/content/bi-analytics-tp`, lo reutiliza. El token únicamente es necesario para clonar el repositorio privado.

In [1]:
from pathlib import Path
import os, sys, subprocess, base64

REPO = "https://github.com/matiasjuarez95/bi-analytics-tp.git"
RAMA = "matias"
PROJECT_DIR = Path("/content/bi-analytics-tp")

if not PROJECT_DIR.is_dir():
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception as exc:
        raise RuntimeError("Agregá GITHUB_TOKEN en Secrets de Colab y habilitá el acceso del notebook.") from exc
    if not token:
        raise RuntimeError("Falta el secret GITHUB_TOKEN en Colab.")
    auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    env = os.environ.copy()
    env["GIT_CONFIG_COUNT"] = "1"
    env["GIT_CONFIG_KEY_0"] = "http.extraheader"
    env["GIT_CONFIG_VALUE_0"] = f"AUTHORIZATION: basic {auth}"
    subprocess.run(["git", "clone", "--quiet", "--branch", RAMA, REPO, str(PROJECT_DIR)],
                   check=True, env=env)
    del token, auth, env
else:
    print("Se reutiliza el repositorio existente; reiniciá el entorno para clonar una versión nueva.")

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "-r", "requirements.txt"], check=True)
from src.etl import procesar_datos as etl
import pandas as pd
from IPython.display import display
print("Repositorio:", PROJECT_DIR)
print("Rama:", subprocess.check_output(["git", "branch", "--show-current"], text=True).strip())

Repositorio: /content/bi-analytics-tp
Rama: matias


## 1. Leer los Excel directamente desde Google Drive

Los datos de entrada están en esta [carpeta compartida](https://drive.google.com/drive/folders/1XUbw9IHaKuRVce5qrULsp31V2HRnQI9W).

Esta celda abre la autenticación de Google, accede **por el ID de la carpeta** (incluso si está en «Compartidos conmigo») y copia los archivos `.xlsx` al almacenamiento temporal de Colab, conservando las subcarpetas. Así, el ETL puede abrirlos con `pandas.read_excel`. No hace falta descargar ni subir ZIP, ni crear un acceso directo en «Mi unidad».

**Requisito:** autenticarse con una cuenta de Google que tenga permiso de lectura sobre la carpeta. Estos archivos se descargan solo a la sesión temporal de Colab; no se incorporan al repositorio de GitHub. Al reiniciar el entorno, habrá que volver a ejecutar esta celda.

In [2]:
from pathlib import Path
from io import BytesIO
from google.colab import auth
import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import re

# Es el ID de la carpeta de Drive, no una ruta del sistema de archivos.
DRIVE_FOLDER_ID = "1XUbw9IHaKuRVce5qrULsp31V2HRnQI9W"
RAW_DIR = Path("/content/datos_lms")
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Autenticación de solo lectura con la cuenta Google que tiene acceso a la carpeta.
auth.authenticate_user()
credenciales, _ = google.auth.default(
    scopes=["https://www.googleapis.com/auth/drive.readonly"]
)
drive = build("drive", "v3", credentials=credenciales, cache_discovery=False)

def listar_hijos(folder_id):
    archivos = []
    pagina = None
    while True:
        respuesta = drive.files().list(
            q=f"'{folder_id}' in parents and trashed = false",
            fields="nextPageToken,files(id,name,mimeType)",
            pageSize=1000,
            pageToken=pagina,
            supportsAllDrives=True,
            includeItemsFromAllDrives=True
        ).execute()
        archivos.extend(respuesta.get("files", []))
        pagina = respuesta.get("nextPageToken")
        if not pagina:
            break
    return sorted(archivos, key=lambda x: x["name"])

def descargar_carpeta(folder_id, destino):
    cantidad = 0
    for elemento in listar_hijos(folder_id):
        nombre = elemento["name"]
        # No permitimos rutas absolutas ni componentes de directorio en los nombres remotos.
        if nombre in (".", "..") or "/" in nombre or chr(92) in nombre:
            raise ValueError(f"Nombre inseguro en Drive: {nombre}")
        ruta = destino / nombre
        if elemento["mimeType"] == "application/vnd.google-apps.folder":
            ruta.mkdir(parents=True, exist_ok=True)
            cantidad += descargar_carpeta(elemento["id"], ruta)
        elif nombre.lower().endswith(".xlsx"):
            if not ruta.exists():
                solicitud = drive.files().get_media(fileId=elemento["id"], supportsAllDrives=True)
                with ruta.open("wb") as salida:
                    downloader = MediaIoBaseDownload(salida, solicitud)
                    terminado = False
                    while not terminado:
                        _, terminado = downloader.next_chunk()
            cantidad += 1
    return cantidad

cantidad = descargar_carpeta(DRIVE_FOLDER_ID, RAW_DIR)
print(f"Excel disponibles: {cantidad}")
print("Carpeta local para el ETL:", RAW_DIR)

# Comprueba que la carpeta incluya los reportes obligatorios de las 3 materias.
archivos = etl.encontrar_archivos(str(RAW_DIR))
print("Asignaturas detectadas:", ", ".join(archivos.keys()))

Excel disponibles: 26
Carpeta local para el ETL: /content/datos_lms
Asignaturas detectadas: Álgebra, Algoritmos y Estructuras de Datos, Probabilidad y Estadística


## 2. Inventario: ¿qué archivos usa cada asignatura?

`encontrar_archivos()` reconoce nombres con y sin tildes, verifica los tres tipos de reporte y la matrícula, y detiene el proceso si encuentra una fuente obligatoria ausente o ambigua.

In [3]:
etl.reiniciar_registro()
archivos = etl.encontrar_archivos(str(RAW_DIR))
inventario = []
for materia, fuentes in archivos.items():
    inventario.append({
        "asignatura": materia,
        "matricula": Path(fuentes["matricula"]).name,
        "participacion": Path(fuentes["participacion"]).name,
        "trazabilidad": Path(fuentes["trazabilidad"]).name,
        "archivos_calificaciones": len(fuentes["calificaciones"]),
    })
display(pd.DataFrame(inventario))
print("Total de Excel detectados:", sum(3 + len(v["calificaciones"]) for v in archivos.values()))

,asignatura,matricula,participacion,trazabilidad,archivos_calificaciones
0,Álgebra,Álgebra.xlsx,reporte_participacion_20251024_115228.xlsx,reporte_trazabilidad_20251024_115226.xlsx,3
1,Algoritmos y Estructuras de Datos,Algoritmos y Estructuras de Datos.xlsx,reporte_participacion_20251024_115446.xlsx,reporte_trazabilidad_20251024_115443.xlsx,5
2,Probabilidad y Estadística,Probabilidad y Estadística.xlsx,reporte_participacion_20251024_115203.xlsx,reporte_trazabilidad_20251024_115157.xlsx,9


Total de Excel detectados: 26


## 3. Matrícula: altas, bajas y re-inscripciones

Una persona puede tener varios movimientos. `leer_matricula()` interpreta el archivo; `resolver_matricula()` conserva **una fila vigente por estudiante y materia**, y mantiene todos los movimientos para auditoría.

**Pregunta para explorar:** ¿qué registros quedaron en Baja y cuáles en Aceptada?

In [4]:
mats, movimientos = [], []
for asig in etl.ASIGNATURAS:
    vig, mov = etl.resolver_matricula(etl.leer_matricula(archivos[asig]["matricula"], asig), asig)
    mats.append(vig)
    movimientos.append(mov)
matricula = pd.concat(mats, ignore_index=True)
matricula_movimientos = pd.concat(movimientos, ignore_index=True)
display(matricula.head(8))
display(pd.crosstab(matricula["asignatura"], matricula["estado_insc"]))
print("Personas con varios movimientos:", matricula_movimientos.loc[
    matricula_movimientos["n_movimientos"] > 1, ["asignatura","id_estudiante"]
].drop_duplicates().shape[0])
display(matricula_movimientos.loc[matricula_movimientos["n_movimientos"] > 1,
    ["asignatura","id_estudiante","estado_insc","fecha_inscripcion","es_vigente"]].head(12))

,nombre_completo,id_estudiante,comision,estado_insc,instancia,fecha_inscripcion,nro_transaccion,asignatura,n_movimientos,caso_ambiguo,matriculado
0,"GÓMEZ, MANUEL",ALU_1001,2319.0,Aceptada,Regularidad,2025-08-15 10:30:08,300846.0,Álgebra,1,False,True
1,"SILVA, DIEGO",ALU_1002,2319.0,Aceptada,Regularidad,2025-09-24 13:13:50,303620.0,Álgebra,1,False,True
2,"FLORES, GABRIEL",ALU_1004,2319.0,Aceptada,Regularidad,2025-07-31 11:32:48,298539.0,Álgebra,1,False,True
3,"RAMOS, PATRICIO",ALU_1006,2319.0,Aceptada,Regularidad,2025-08-04 22:01:50,299619.0,Álgebra,1,False,True
4,"RAMÍREZ, GABRIEL",ALU_1008,2319.0,Aceptada,Regularidad,2025-08-06 18:44:34,300212.0,Álgebra,1,False,True
5,"BENÍTEZ, MATÍAS",ALU_1013,2319.0,Aceptada,Regularidad,2025-08-04 00:34:44,299230.0,Álgebra,1,False,True
6,"FERRER, JULIETA",ALU_1016,2319.0,Aceptada,Regularidad,2025-09-24 18:24:38,303671.0,Álgebra,1,False,True
7,"RODRÍGUEZ, FACUNDO",ALU_1018,2319.0,Aceptada,Regularidad,2025-08-05 09:28:16,299745.0,Álgebra,1,False,True


estado_insc,Aceptada,Baja
asignatura,,
Algoritmos y Estructuras de Datos,165,2
Probabilidad y Estadística,164,3
Álgebra,163,2


Personas con varios movimientos: 11


,asignatura,id_estudiante,estado_insc,fecha_inscripcion,es_vigente
112,Álgebra,ALU_1193,Baja,2025-07-28 23:20:36,False
113,Álgebra,ALU_1193,Baja,2025-07-31 08:54:37,False
114,Álgebra,ALU_1193,Baja,2025-08-04 17:02:02,True
181,Algoritmos y Estructuras de Datos,ALU_1028,Baja,2025-08-02 16:06:52,False
182,Algoritmos y Estructuras de Datos,ALU_1028,Aceptada,2025-08-03 00:31:46,True
185,Algoritmos y Estructuras de Datos,ALU_1034,Baja,2025-08-14 10:42:55,False
186,Algoritmos y Estructuras de Datos,ALU_1034,Aceptada,2025-08-14 21:48:05,True
202,Algoritmos y Estructuras de Datos,ALU_1064,Baja,2025-08-13 20:20:55,False
203,Algoritmos y Estructuras de Datos,ALU_1064,Aceptada,2025-08-13 20:21:21,True
217,Algoritmos y Estructuras de Datos,ALU_1084,Baja,2025-08-08 13:21:41,False


## 4. Participación: ¿quién ingresó y cuánto participó?

Las fechas, los tiempos de uso y las interacciones se normalizan. Se separan los perfiles de alumnos del personal y se detectan registros con cero interacción.

In [5]:
participaciones, personal_participacion = [], []
for asig in etl.ASIGNATURAS:
    df = etl.leer_participacion(archivos[asig]["participacion"], asig)
    alumnos, personal = etl.procesar_participacion(df, asig)
    participaciones.append(alumnos)
    personal_participacion.append(personal)
participacion = pd.concat(participaciones, ignore_index=True)
display(participacion[["asignatura","id_estudiante","interacciones",
                        "tiempo_uso_min","nunca_ingreso"]].head(10))
display(participacion.groupby("asignatura", as_index=False).agg(
    estudiantes=("id_estudiante","nunique"),
    interacciones=("interacciones","sum"),
    nunca_ingresaron=("nunca_ingreso","sum"),
    tiempo_medio_min=("tiempo_uso_min","mean")
).round(1))

,asignatura,id_estudiante,interacciones,tiempo_uso_min,nunca_ingreso
0,Álgebra,ALU_1001,1015.0,607.0,False
1,Álgebra,ALU_1002,616.0,389.0,False
2,Álgebra,ALU_1003,0.0,0.0,True
3,Álgebra,ALU_1004,633.0,514.0,False
4,Álgebra,ALU_1005,410.0,300.0,False
5,Álgebra,ALU_1006,730.0,671.0,False
6,Álgebra,ALU_1007,0.0,0.0,True
7,Álgebra,ALU_1008,244.0,283.0,False
8,Álgebra,ALU_1009,177.0,231.0,False
9,Álgebra,ALU_1010,30.0,34.0,False


,asignatura,estudiantes,interacciones,nunca_ingresaron,tiempo_medio_min
0,Algoritmos y Estructuras de Datos,319,55494.0,43,137.5
1,Probabilidad y Estadística,319,78444.0,46,264.0
2,Álgebra,319,56456.0,51,151.5


## 5. Trazabilidad: eventos, duplicados y tipos de interacción

A diferencia de la participación, la trazabilidad contiene **muchos eventos por estudiante**. El ETL conserva eventos idénticos y los marca como duplicados *potenciales*; no asume que sean errores.

In [6]:
trazas, personal_trazabilidad = [], []
for asig in etl.ASIGNATURAS:
    tr, n_original = etl.leer_trazabilidad(archivos[asig]["trazabilidad"], asig)
    alumnos, personal = etl.separar_alumnos_personal(tr, "trazabilidad", asig)
    trazas.append(alumnos)
    personal_trazabilidad.append(personal)
trazabilidad = pd.concat(trazas, ignore_index=True)
display(trazabilidad[["asignatura","id_estudiante","fecha","accion",
                       "tipo_interaccion","duplicado_potencial"]].head(10))
display(trazabilidad.groupby(["asignatura","tipo_interaccion"]).size().unstack(fill_value=0))
print("Eventos potencialmente repetidos:", int(trazabilidad["duplicado_potencial"].sum()))

,asignatura,id_estudiante,fecha,accion,tipo_interaccion,duplicado_potencial
0,Álgebra,ALU_1001,2025-08-05 10:50:31,Acceder,Acceso/navegación,False
1,Álgebra,ALU_1001,2025-08-11 13:29:48,Acceder,Acceso/navegación,False
2,Álgebra,ALU_1001,2025-08-11 13:29:57,Acceder,Acceso/navegación,False
3,Álgebra,ALU_1001,2025-08-11 13:30:01,Ver tópico,Lectura de materiales,False
4,Álgebra,ALU_1001,2025-08-11 13:30:35,Acceder,Acceso/navegación,False
5,Álgebra,ALU_1001,2025-08-11 13:30:45,Ver texto,Lectura de materiales,False
6,Álgebra,ALU_1001,2025-08-11 13:31:30,Ver tema,Lectura de materiales,False
7,Álgebra,ALU_1001,2025-08-11 13:31:51,Ver participacion,Otros,False
8,Álgebra,ALU_1001,2025-08-11 13:32:14,Ver tema,Lectura de materiales,False
9,Álgebra,ALU_1001,2025-08-11 13:36:30,Ver texto,Lectura de materiales,False


tipo_interaccion,Acceso/navegación,Calificaciones,Comunicación/foros,Evaluaciones,Lectura de materiales,Otros
asignatura,,,,,,
Algoritmos y Estructuras de Datos,18012,469,318,2357,21676,1711
Probabilidad y Estadística,24392,1171,940,6583,29431,1061
Álgebra,18824,674,435,3923,16661,1466


Eventos potencialmente repetidos: 646


In [7]:
import plotly.express as px
tipo_por_materia = (trazabilidad.groupby(["asignatura","tipo_interaccion"])
                    .size().reset_index(name="eventos"))
fig = px.bar(tipo_por_materia, x="asignatura", y="eventos",
             color="tipo_interaccion", barmode="stack",
             title="Eventos del LMS por tipo y asignatura")
fig.show()

## 6. Calificaciones: formato largo y recuperatorios

Cada archivo Excel representa un instrumento. El ETL lee los archivos, extrae nota y máximo, y genera **una fila por estudiante e instrumento**.

Los recuperatorios tienen una lógica particular: el código señala cuáles notas se usan y cuáles no, pero no elimina los registros históricos.

In [8]:
calificaciones_list = []
for asig in etl.ASIGNATURAS:
    for ruta in archivos[asig]["calificaciones"]:
        calificaciones_list.append(etl.leer_calificaciones(ruta, asig))
calificaciones_crudas = pd.concat(calificaciones_list, ignore_index=True)
calificaciones = etl.aplicar_recuperatorios(calificaciones_crudas, modo="docente")
display(calificaciones[["id_estudiante","asignatura","instrumento","puntos","maximo",
                         "nota_100","es_recuperatorio","usa_en_nota_final"]].head(12))
display(calificaciones.groupby(["asignatura","tipo_instrumento"]).agg(
    registros=("id_estudiante","size"),
    notas_utilizadas=("usa_en_nota_final","sum")
).reset_index())
display(calificaciones.loc[calificaciones["es_recuperatorio"],
     ["id_estudiante","instrumento","nota_100","usa_en_nota_final"]].head(15))

/content/bi-analytics-tp/src/etl/procesar_datos.py:491: FutureWarning:

Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`



,id_estudiante,asignatura,instrumento,puntos,maximo,nota_100,es_recuperatorio,usa_en_nota_final
0,ALU_1001,Álgebra,AUTOEVALUACION1,9.0,10.0,90.0,False,True
1,ALU_1004,Álgebra,AUTOEVALUACION1,9.0,10.0,90.0,False,True
2,ALU_1005,Álgebra,AUTOEVALUACION1,9.0,10.0,90.0,False,True
3,ALU_1006,Álgebra,AUTOEVALUACION1,10.0,10.0,100.0,False,True
4,ALU_1008,Álgebra,AUTOEVALUACION1,10.0,10.0,100.0,False,True
5,ALU_1009,Álgebra,AUTOEVALUACION1,10.0,10.0,100.0,False,True
6,ALU_1012,Álgebra,AUTOEVALUACION1,9.0,10.0,90.0,False,True
7,ALU_1013,Álgebra,AUTOEVALUACION1,9.0,10.0,90.0,False,True
8,ALU_1015,Álgebra,AUTOEVALUACION1,9.0,10.0,90.0,False,True
9,ALU_1018,Álgebra,AUTOEVALUACION1,7.0,10.0,70.0,False,True


,asignatura,tipo_instrumento,registros,notas_utilizadas
0,Algoritmos y Estructuras de Datos,Autoevaluación,419,419
1,Probabilidad y Estadística,Actividad/Evaluación,707,701
2,Probabilidad y Estadística,Parcial,167,167
3,Probabilidad y Estadística,Recuperatorio,21,14
4,Álgebra,Autoevaluación,317,317
5,Álgebra,Parcial,162,162


,id_estudiante,instrumento,nota_100,usa_en_nota_final
1349,ALU_1020,Recuperatorio: Evaluación Semanal Nº1,0.00,True
1350,ALU_1035,Recuperatorio: Evaluación Semanal Nº1,0.00,False
1351,ALU_1044,Recuperatorio: Evaluación Semanal Nº1,83.33,True
1352,ALU_1055,Recuperatorio: Evaluación Semanal Nº1,94.44,True
1353,ALU_1063,Recuperatorio: Evaluación Semanal Nº1,94.44,False
1354,ALU_1076,Recuperatorio: Evaluación Semanal Nº1,77.78,True
1355,ALU_1079,Recuperatorio: Evaluación Semanal Nº1,83.33,True
1356,ALU_1092,Recuperatorio: Evaluación Semanal Nº1,88.89,True
1357,ALU_1105,Recuperatorio: Evaluación Semanal Nº1,0.00,False
1358,ALU_1118,Recuperatorio: Evaluación Semanal Nº1,83.33,True


## 7. Universo de alumnos: integración sin duplicar estudiantes

El universo es la unión de quienes aparecen en matrícula, participación, trazabilidad o calificaciones, usando `(id_estudiante, asignatura)` como clave.

Un alumno puede estar en varias materias: lo contamos una vez **por materia**, no una vez por evento.

In [9]:
universo = pd.concat([
    matricula[["id_estudiante","asignatura"]],
    participacion[["id_estudiante","asignatura"]],
    calificaciones[["id_estudiante","asignatura"]],
    trazabilidad[["id_estudiante","asignatura"]],
], ignore_index=True).drop_duplicates().reset_index(drop=True)
print("Filas del universo:", len(universo))
print("Claves duplicadas:", universo.duplicated(["id_estudiante","asignatura"]).sum())
display(universo.groupby("asignatura").size().rename("alumnos").reset_index())

Filas del universo: 957
Claves duplicadas: 0


,asignatura,alumnos
0,Algoritmos y Estructuras de Datos,319
1,Probabilidad y Estadística,319
2,Álgebra,319


## 8. Notas finales: real vs. estimado

**Álgebra** acumula puntos reales, pero también calcula una proyección sobre los instrumentos disponibles. Las proyecciones **no determinan el estado definitivo**.

**Probabilidad y Algoritmos** usan promedios; el ETL distingue promedio observado y promedio considerando cero para exámenes presentes que no se rindieron.

El `estado_provisorio` y el `estado_definitivo` no significan lo mismo.

In [10]:
notas_finales = etl.calcular_notas_finales(calificaciones, universo, cursada_cerrada=False)
columnas = ["asignatura","id_estudiante","instrumentos_rendidos","nota_a_la_fecha",
            "estado_provisorio","estado_definitivo"]
display(notas_finales[columnas].head(12))
algebra = notas_finales[notas_finales["asignatura"] == "Álgebra"]
display(algebra[["id_estudiante","puntos_acumulados","puntos_maximos_observados",
                 "proyeccion_120_observados","estado_provisorio","estado_definitivo"]].head(12))
display(pd.crosstab(notas_finales["asignatura"], notas_finales["estado_provisorio"]))

,asignatura,id_estudiante,instrumentos_rendidos,nota_a_la_fecha,estado_provisorio,estado_definitivo
0,Álgebra,ALU_1001,3,103.55,Promociona,Pendiente
1,Álgebra,ALU_1002,2,72.00,Regular,Pendiente
2,Álgebra,ALU_1004,3,115.12,Promociona,Pendiente
3,Álgebra,ALU_1006,3,107.12,Promociona,Pendiente
4,Álgebra,ALU_1008,3,98.67,Promociona,Pendiente
5,Álgebra,ALU_1013,3,106.67,Promociona,Pendiente
6,Álgebra,ALU_1016,0,NaN,Sin notas,Sin notas
7,Álgebra,ALU_1018,3,96.88,Promociona,Pendiente
8,Álgebra,ALU_1019,3,106.67,Promociona,Pendiente
9,Álgebra,ALU_1020,0,NaN,Sin notas,Sin notas


,id_estudiante,puntos_acumulados,puntos_maximos_observados,proyeccion_120_observados,estado_provisorio,estado_definitivo
0,ALU_1001,38.83,45.0,103.55,Promociona,Pendiente
1,ALU_1002,27.00,45.0,72.00,Regular,Pendiente
2,ALU_1004,43.17,45.0,115.12,Promociona,Pendiente
3,ALU_1006,40.17,45.0,107.12,Promociona,Pendiente
4,ALU_1008,37.00,45.0,98.67,Promociona,Pendiente
5,ALU_1013,40.00,45.0,106.67,Promociona,Pendiente
6,ALU_1016,NaN,45.0,NaN,Sin notas,Sin notas
7,ALU_1018,36.33,45.0,96.88,Promociona,Pendiente
8,ALU_1019,40.00,45.0,106.67,Promociona,Pendiente
9,ALU_1020,NaN,45.0,NaN,Sin notas,Sin notas


estado_provisorio,Libre,Promociona,Regular,Sin notas
asignatura,,,,
Algoritmos y Estructuras de Datos,81,22,43,173
Probabilidad y Estadística,100,19,75,125
Álgebra,45,124,27,123


## 9. Tabla integrada: situación de cada estudiante

El ETL agrega por estudiante-materia los eventos, días activos, primer y último evento; luego combina todo con matrícula, participación y notas.

**Regla clave:** `situacion_estudiante` debe tener exactamente una fila por `(id_estudiante, asignatura)`.

In [11]:
eventos = trazabilidad.groupby(["id_estudiante","asignatura"]).agg(
    eventos_trazabilidad=("fecha","size"),
    eventos_repetidos_potenciales=("duplicado_potencial","sum"),
    dias_activos=("fecha_dia","nunique"),
    primer_evento=("fecha","min"),
    ultimo_evento=("fecha","max"),
).reset_index()
situacion = etl.construir_situacion(universo, matricula, participacion, eventos, notas_finales)
print("Filas del universo:",len(universo),"| Filas del modelo:",len(situacion))
assert not situacion.duplicated(["id_estudiante","asignatura"]).any()
display(situacion[["asignatura","id_estudiante","matriculado",
                   "tiene_actividad","situacion_matricula","estado_provisorio","semaforo"]].head(15))
display(pd.crosstab(situacion["asignatura"], situacion["situacion_matricula"]))

Filas del universo: 957 | Filas del modelo: 957


,asignatura,id_estudiante,matriculado,tiene_actividad,situacion_matricula,estado_provisorio,semaforo
0,Álgebra,ALU_1001,True,True,Matriculado con actividad,Promociona,Verde
1,Álgebra,ALU_1002,True,True,Matriculado con actividad,Regular,Amarillo
2,Álgebra,ALU_1004,True,True,Matriculado con actividad,Promociona,Verde
3,Álgebra,ALU_1006,True,True,Matriculado con actividad,Promociona,Verde
4,Álgebra,ALU_1008,True,True,Matriculado con actividad,Promociona,Verde
5,Álgebra,ALU_1013,True,True,Matriculado con actividad,Promociona,Verde
6,Álgebra,ALU_1016,True,True,Matriculado con actividad,Sin notas,Sin notas
7,Álgebra,ALU_1018,True,True,Matriculado con actividad,Promociona,Verde
8,Álgebra,ALU_1019,True,True,Matriculado con actividad,Promociona,Verde
9,Álgebra,ALU_1020,True,True,Matriculado con actividad,Sin notas,Sin notas


situacion_matricula,Matriculado con actividad,Matriculado sin participación,NO matriculado con actividad,No matriculado sin actividad
asignatura,,,,
Algoritmos y Estructuras de Datos,163,2,113,41
Probabilidad y Estadística,161,3,112,43
Álgebra,161,2,107,49


## 10. Calidad: ¿qué se descartó y qué quedó marcado?

El registro de calidad `LOG` guarda decisiones, afectados y balances. `DESCARTADOS` conserva filas descartadas. Mirá las causas antes de confiar en una métrica.

In [12]:
calidad = pd.DataFrame(etl.LOG)
descartados = (pd.concat(etl.DESCARTADOS, ignore_index=True) if etl.DESCARTADOS
              else pd.DataFrame(columns=["fuente","asignatura","motivo"]))
print("Registros de calidad:",len(calidad),"| Filas descartadas:",len(descartados))
display(calidad[["fuente","asignatura","problema","registros_afectados","decision"]].head(25))
display(descartados.head(10))

Registros de calidad: 66 | Filas descartadas: 3


,fuente,asignatura,problema,registros_afectados,decision
0,matricula,Álgebra,Filas sin ID (pie 'Cantidad de filas' / vacías),1,Descartadas (quedan en descartados.csv)
1,matricula,Álgebra,IDs con más de un movimiento (altas/bajas),1,Estado vigente = movimiento más reciente (dese...
2,matricula,Álgebra,"de ellos, bajas seguidas de una nueva inscri...",0,Quedan matriculados (el último movimiento es A...
3,matricula,Álgebra,"de ellos, casos ambiguos (fecha/nro. transac...",0,Revisar manualmente: ninguno
4,matricula,Álgebra,Alumnos cuyo estado vigente es 'Baja',2,matriculado=False (no cuentan como inscriptos)
5,matricula,Álgebra,Nombres repetidos con IDs distintos (posibles ...,9,Se tratan como personas distintas: la clave es...
6,matricula,Algoritmos y Estructuras de Datos,Filas sin ID (pie 'Cantidad de filas' / vacías),1,Descartadas (quedan en descartados.csv)
7,matricula,Algoritmos y Estructuras de Datos,IDs con más de un movimiento (altas/bajas),7,Estado vigente = movimiento más reciente (dese...
8,matricula,Algoritmos y Estructuras de Datos,"de ellos, bajas seguidas de una nueva inscri...",5,Quedan matriculados (el último movimiento es A...
9,matricula,Algoritmos y Estructuras de Datos,"de ellos, casos ambiguos (fecha/nro. transac...",0,Revisar manualmente: ninguno


,nombre_completo,id_estudiante,comision,estado_insc,instancia,fecha_inscripcion,nro_transaccion,fuente,asignatura,motivo
0,Cantidad de filas:,NaN,NaN,NaN,NaN,NaN,NaN,matricula,Álgebra,Fila sin ID (pie 'Cantidad de filas' u otra no...
1,Cantidad de filas:,NaN,NaN,NaN,NaN,NaN,NaN,matricula,Algoritmos y Estructuras de Datos,Fila sin ID (pie 'Cantidad de filas' u otra no...
2,Cantidad de filas:,NaN,NaN,NaN,NaN,NaN,NaN,matricula,Probabilidad y Estadística,Fila sin ID (pie 'Cantidad de filas' u otra no...


## 11. Validar el modelo

Esta celda ejecuta las comprobaciones que ya implementó Lara: claves únicas, rangos de notas, semáforos, matrícula e integridad de la integración. Una validación `ok=False` requiere análisis antes de avanzar a los KPI.

In [13]:
personal = pd.concat(personal_participacion + personal_trazabilidad, ignore_index=True)
validaciones = etl.validar_modelo(
    situacion, notas_finales, calificaciones, matricula, universo, personal,
    cursada_cerrada=False
)
display(validaciones)
assert validaciones["ok"].all(), "Existen validaciones fallidas: revisar antes de continuar."

,chequeo,ok,detalle
0,situacion_estudiante única por id_estudiante+a...,True,957 filas
1,los merges no multiplican filas (situación = u...,True,universo=957 situación=957
2,notas_finales única por id_estudiante+asignatura,True,
3,matrícula vigente única por id_estudiante+asig...,True,
4,notas dentro del rango válido (0..máximo),True,0 fuera de rango
5,'Sin notas' no termina como Libre/Regular/Prom...,True,421 alumnos sin notas
6,con la cursada abierta no hay Regular/Libre de...,True,0 casos
7,el personal excluido no incluye alumnos matric...,True,0 coincidencias
8,[informativo] matriculados sin participación,True,7
9,[informativo] actividad sin matrícula,True,332


## 12. Ejecutar exactamente el ETL completo (punta a punta)

Hasta acá inspeccionamos cada paso con DataFrames en memoria. Ahora llamamos al orquestador oficial `etl.ejecutar()`, que reproduce el pipeline de producción y escribe las tablas procesadas.

**Nota:** puede generar CSV (y opcionalmente Parquet). Eso no convierte ni modifica las **fuentes originales Excel**.

In [14]:
SALIDA = Path("/content/etl_resultados")
salidas = etl.ejecutar(str(RAW_DIR), str(SALIDA))
resumen_salidas = pd.DataFrame([{"tabla":nombre, "filas":len(df), "columnas":len(df.columns)}
                                for nombre,df in salidas.items()])
display(resumen_salidas)
print("Archivos generados:",sorted(p.name for p in SALIDA.iterdir()))

[Álgebra] leyendo...
[Algoritmos y Estructuras de Datos] leyendo...
[Probabilidad y Estadística] leyendo...


/content/bi-analytics-tp/src/etl/procesar_datos.py:491: FutureWarning:

Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`



   matricula: 499 filas
   matricula_movimientos: 513 filas
   participacion: 957 filas
   trazabilidad: 150,104 filas
   calificaciones: 1,793 filas
   notas_finales: 957 filas
   situacion_estudiante: 957 filas
   personal_excluido: 9,729 filas
   descartados: 3 filas
   informe_calidad_datos: 91 filas

Validaciones:
   [OK] situacion_estudiante única por id_estudiante+asignatura - 957 filas
   [OK] los merges no multiplican filas (situación = universo) - universo=957 situación=957
   [OK] notas_finales única por id_estudiante+asignatura 
   [OK] matrícula vigente única por id_estudiante+asignatura 
   [OK] notas dentro del rango válido (0..máximo) - 0 fuera de rango
   [OK] 'Sin notas' no termina como Libre/Regular/Promociona - 421 alumnos sin notas
   [OK] con la cursada abierta no hay Regular/Libre definitivos ni Promociona sin puntos reales suficientes - 0 casos
   [OK] el personal excluido no incluye alumnos matriculados - 0 coincidencias
   [OK] [informativo] matriculados sin p

,tabla,filas,columnas
0,matricula,499,11
1,matricula_movimientos,513,12
2,participacion,957,14
3,trazabilidad,150104,13
4,calificaciones,1793,14
5,notas_finales,957,21
6,situacion_estudiante,957,48
7,personal_excluido,9729,25
8,descartados,3,10
9,informe_calidad_datos,91,7


Archivos generados: ['calificaciones.csv', 'calificaciones.parquet', 'descartados.csv', 'descartados.parquet', 'informe_calidad_datos.csv', 'informe_calidad_datos.parquet', 'matricula.csv', 'matricula.parquet', 'matricula_movimientos.csv', 'matricula_movimientos.parquet', 'notas_finales.csv', 'notas_finales.parquet', 'participacion.csv', 'participacion.parquet', 'personal_excluido.csv', 'personal_excluido.parquet', 'situacion_estudiante.csv', 'situacion_estudiante.parquet', 'trazabilidad.csv', 'trazabilidad.parquet', 'validaciones.csv']


## 13. Pruebas automatizadas

Además de la exploración visual, ejecutá los tests existentes con datos sintéticos. Estos tests verifican reglas y casos límite **sin depender de los Excel**.

In [15]:
resultado = subprocess.run([sys.executable,"-m","pytest","tests/test_etl_6_1.py","-q"],
                            cwd=str(PROJECT_DIR), text=True)
if resultado.returncode:
    raise RuntimeError(f"Fallaron las pruebas (código {resultado.returncode}).")

## Qué sigue

Este notebook es un **laboratorio de lectura y exploración**, no otro ETL distinto. La lógica reside únicamente en `src/etl/procesar_datos.py`.

Para futuras correcciones, modificá los scripts en tu rama, hacé `push` y luego **reiniciá el runtime de Colab** para clonar la versión nueva. El notebook no modifica las fuentes ni hace `push` a GitHub.

**Atención:** la columna `semaforo_definitivo` del ETL actual devuelve un booleano y tiene una corrección pendiente en nuestra rama. No confundas esa columna con el semáforo de colores `semaforo`.